## Constructing PyTorch Geometric Graph Objects

**Background**  
In previous notebooks, we optimized pairwise distance calculations ($\mathbf{D}_{ij}$) and established that sparse COO edge lists ($\mathbb{Z}^{2 \times E}$) dramatically reduce memory footprint compared to dense matrices. However, message-passing Graph Neural Networks (GNNs) require more than connectivity alone—they operate on structured graph containers that pair topological edge indices with spatial, sequence, and chemical feature attributes. In this notebook, we integrate our distance and sparsity pipelines to construct fully featured, mathematically validated `torch_geometric.data.Data` objects for **Par-6** and **Lgl**.

**Goals**  
1. **Node Feature Engineering ($\mathbf{X} \in \mathbb{R}^{N \times d_{\text{node}}}$):** Construct representations for each $C_\alpha$ residue node (e.g., one-hot encoded amino acid identity, normalized sequence position).
2. **Geometric Edge Attributes ($\mathbf{E}_{\text{feat}} \in \mathbb{R}^{E \times d_{\text{edge}}}$):** Enrich the sparse COO `edge_index` ($\mathbb{Z}^{2 \times E}$) with continuous distance features, directional unit vectors, and Gaussian Radial Basis Function (RBF) expansions.
3. **PyG Data Assembly:** Package node features, coordinates, edge topology, edge attributes, and graph metadata into standardized `torch_geometric.data.Data` containers.
4. **Graph Integrity Validation:** Implement automated assertions to verify tensor shape compatibility, absence of `NaN`/`Inf` values, and undirected edge symmetry before message passing.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
import torch.nn as nn

from pathlib import Path
from torch_geometric.data import Data

from polarity_engine.parsers import StructureParser

/Users/chiharugraybill/aPKC-mechanics-analysis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data_dir = Path("../../data")
data_dir.mkdir(parents=True, exist_ok=True)

input_dir = data_dir / "input"
input_dir.mkdir(parents=True, exist_ok=True)

In [4]:
TEST_CIF_Paht = input_dir/ "8r3y.cif"
test_coords = StructureParser.get_alpha_carbon_coordinates(TEST_CIF_Paht)

In [5]:
AMINO_ACID_TO_INDEX = {
    'ALA': 0,
    'ARG': 1,
    'ASN': 2,  
    'ASP': 3,
    'CYS': 4,
    'GLN': 5,
    'GLU': 6,
    'GLY': 7,
    'HIS': 8,
    'ILE': 9,
    'LEU': 10,
    'LYS': 11,
    'MET': 12,
    'PHE': 13,
    'PRO': 14,
    'SER': 15,
    'THR': 16,
    'TRP': 17,
    'TYR': 18,
    'VAL': 19,
    'UNK': 20  # Unknown / Non-standard
}


### Step 1: Construct feature vectors for each residue node ($\mathbf{X} \in \mathbb{R}^{N \times d_{\text{node}}}$)

In [6]:
node_features_dict = {}

# Iterate over parsed structures directly
for name, data in test_coords.items():
    aa_list = data["aa_residues"]

    # Map residue strings to integer indices safely
    aa_indices = torch.tensor(
        [AMINO_ACID_TO_INDEX.get(
            res.upper(), AMINO_ACID_TO_INDEX["UNK"]) for res in aa_list],
        dtype=torch.long,
    )

    # One-hot encode to float32 tensor (N, 21)
    x_one_hot = torch.nn.functional.one_hot(
        aa_indices, num_classes=21
    ).float()

    # Create normalized sequence positions in range [0, 1]
    seq_pos = torch.arange(
        len(aa_list), dtype=torch.float32) / max(len(aa_list) - 1, 1)

    seq_pos_col = seq_pos.unsqueeze(1)

    x_enriched = torch.cat([x_one_hot, seq_pos_col], dim=1)
    node_features_dict[name] = x_enriched

### Step 2: Using the sparse edge_index tensor $\mathbf{E} \in \mathbb{Z}^{2 \times E}$ from notebook 02, compute edge geometric attributes for every active interaction pair $(i, j)$

In [7]:
def distance_to_sparse_coo(
        dist_matrix: torch.Tensor,
        cutoff: float = 8.0,
        include_self_loops: bool = False
) -> tuple[torch.Tensor, int]:
    """
    Converts a dense pairwise distance matrix directly to PyTorch Geometric COO edge_index format.

    Args:
        dist_matrix: (N, N) distance tensor.
        cutoff: Distance threshold in Angstroms.
        include_self_loops: Whether diagonal elements should be retained.

    Returns:
        edge_index: (2, E) tensor containing source and target node indices.
        num_edges: Total number of active edges E.
    """
    mask = dist_matrix <= cutoff

    if not include_self_loops:
        mask.fill_diagonal_(False)

    # torch.nonzero returns (E, 2) indices -> transpose to (2, E) for PyG standard
    # .contiguous() forces PyTorch to rearrange the data sequentially in memory,
    edge_index = torch.nonzero(mask).t().contiguous()
    num_edges = edge_index.shape[1]

    return edge_index, num_edges

In [8]:
edge_geometric_attrs = {}

for name, data in test_coords.items():
    coords = data["coords"]

    # Convert to float32 tensor
    coords_torch = torch.from_numpy(coords).float()

    # Extract sparse edge topology
    dist_matrix = torch.cdist(coords_torch, coords_torch)
    edge_index, num_edges = distance_to_sparse_coo(dist_matrix)

    # Vectorized coordinate indexing
    pos_i = coords_torch[edge_index[0]]  # (E, 3)
    pos_j = coords_torch[edge_index[1]]  # (E, 3)

    # Geometric features
    disp_vec = pos_j - pos_i  # (E, 3)
    dist_scalar = torch.linalg.vector_norm(disp_vec, dim=-1, keepdim=True)  # (E, 1)
    unit_vec = disp_vec / (dist_scalar + 1e-8)  # (E, 3)

    # Store features for step 3 (RBF expansion & assembly)
    edge_geometric_attrs[name] = {
        "edge_index": edge_index,
        "dist_scalar": dist_scalar,
        "unit_vec": unit_vec,
    }

### Step 3: Expand each scalar distance into $K$ Gaussian basis functions

In [9]:
class GaussianRBF(nn.Module):
    """Gaussian Radial Basis Function expansion module in pure PyTorch."""

    def __init__(
        self, start: float = 0.0, stop: float = 8.0, num_gaussians: int = 16
    ):
        super().__init__()
        # Spaced centers (mu) from start to stop
        mu = torch.linspace(start, stop, num_gaussians)
        # Step size between centers
        delta = (stop - start) / (num_gaussians - 1)
        # Gamma (width parameter) based on step size
        gamma = 1.0 / (delta**2)

        # Register buffers so they move to GPU automatically with module
        self.register_buffer("mu", mu)
        self.register_buffer("gamma", torch.tensor(gamma))

    def forward(self, dist: torch.Tensor) -> torch.Tensor:
        # Ensure shape is (E, 1)
        if dist.dim() == 1:
            dist = dist.unsqueeze(-1)

        # Broadcast against centers: (E, 1) - (1, K) -> (E, K)
        diff = dist - self.mu.unsqueeze(0)
        return torch.exp(-self.gamma * (diff**2))

In [10]:
edge_attrs = {}
rbf_expansion = GaussianRBF(start=0.0, stop=8.0, num_gaussians=16)

for name in test_coords.keys():
    dist_scalar = edge_geometric_attrs[name]["dist_scalar"]
    unit_vec = edge_geometric_attrs[name]["unit_vec"]

    # Inside your loop for each structure:
    # dist_scalar shape: (E, 1) -> rbf_out shape: (E, 16)
    rbf_out = rbf_expansion(dist_scalar)

    # Concatenate with unit_vec (E, 3) along dim=1 -> edge_attr shape: (E, 19)
    edge_attrs[name] = torch.cat([unit_vec, rbf_out], dim=1)

In [11]:
for name, attr in edge_attrs.items():
    print(f"{name} edge_attr shape: {attr.shape}")

I edge_attr shape: torch.Size([3072, 19])
L edge_attr shape: torch.Size([8458, 19])
P edge_attr shape: torch.Size([818, 19])


### Assembling torch_geometric.data.Data Objects

In [12]:
pyg_graph_dataset = {}

for name in test_coords.keys():
    # checking input
    coords_np = test_coords[name]["coords"]
    aa_list = test_coords[name]["aa_residues"]
    if not np.isfinite(coords_np).all():
        raise ValueError(
            f"Invalid coordinates in structure '{name}': input contains NaN or Inf values."
        )
    else:
        print(f"No NaN or Inf are in structure '{name}'")

    if len(aa_list) != len(coords_np):
        raise ValueError(
            f"Length mismatch in structure '{name}': "
            f"got {len(aa_list)} amino acids but {len(coords_np)} coordinate vectors."
        )
    # Extract node features (N, 22) and coordinates (N, 3)
    x_nodes = node_features_dict[name]
    pos_coords = torch.from_numpy(coords_np).float()

    # Extract topology (2, E) and combined edge attributes (E, 19)
    edge_idx = edge_geometric_attrs[name]["edge_index"]
    edge_features = edge_attrs[name]

    # Assemble into PyG Data container
    graph_data = Data(
        x=x_nodes,
        edge_index=edge_idx,
        edge_attr=edge_features,
        pos=pos_coords,
        name=name,
    )

    pyg_graph_dataset[name] = graph_data

No NaN or Inf are in structure 'I'
No NaN or Inf are in structure 'L'
No NaN or Inf are in structure 'P'


In [13]:
for name, sample_graph in pyg_graph_dataset.items():
    print(f"Graph Summary: {sample_graph}")
    print(f"Number of Nodes: {sample_graph.num_nodes}")
    print(f"Number of Edges: {sample_graph.num_edges}")
    print(f"Is Directed?: {sample_graph.is_directed()}")
    print(f"Contains Isolated Nodes?: {sample_graph.has_isolated_nodes()}")
    print(f"Contains Self-Loops?: {sample_graph.has_self_loops()}")

Graph Summary: Data(x=[336, 22], edge_index=[2, 3072], edge_attr=[3072, 19], pos=[336, 3], name='I')
Number of Nodes: 336
Number of Edges: 3072
Is Directed?: True
Contains Isolated Nodes?: False
Contains Self-Loops?: False
Graph Summary: Data(x=[860, 22], edge_index=[2, 8458], edge_attr=[8458, 19], pos=[860, 3], name='L')
Number of Nodes: 860
Number of Edges: 8458
Is Directed?: True
Contains Isolated Nodes?: False
Contains Self-Loops?: False
Graph Summary: Data(x=[94, 22], edge_index=[2, 818], edge_attr=[818, 19], pos=[94, 3], name='P')
Number of Nodes: 94
Number of Edges: 818
Is Directed?: True
Contains Isolated Nodes?: False
Contains Self-Loops?: False
